In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv('garments_worker_productivity.csv')
print("Shape:", df.shape)
print("\nInfo:")
print(df.info())
print("\nMissing values:")
print(df.isnull().sum())
print("\nFirst 5 rows:")
print(df.head())
print("\nUnique values in categorical columns:")
for col in df.select_dtypes(include='object').columns:
    print(col, ":", df[col].unique())

Shape: (1197, 15)

Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1197 entries, 0 to 1196
Data columns (total 15 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   date                   1197 non-null   object 
 1   quarter                1197 non-null   object 
 2   department             1197 non-null   object 
 3   day                    1197 non-null   object 
 4   team                   1197 non-null   int64  
 5   targeted_productivity  1197 non-null   float64
 6   smv                    1197 non-null   float64
 7   wip                    691 non-null    float64
 8   over_time              1197 non-null   int64  
 9   incentive              1197 non-null   int64  
 10  idle_time              1197 non-null   float64
 11  idle_men               1197 non-null   int64  
 12  no_of_style_change     1197 non-null   int64  
 13  no_of_workers          1197 non-null   float64
 14  actual_productivity    1197 non

In [2]:
import pandas as pd
import numpy as np
import time
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score,
    accuracy_score, precision_score, recall_score, f1_score
)

# 1. Load Data & Summary Statistics
df = pd.read_csv('garments_worker_productivity.csv')
stats = df.describe().T[['count', 'mean', '50%', 'min', 'max', 'std']].rename(columns={'50%': 'median'})
print("=== Summary Statistics ===")
print(stats.round(4))

# Target definitions
# Regression target: actual_productivity
# Classification target: Meets Target = 1 when actual_productivity >= targeted_productivity, else 0
df['Meets_Target'] = (df['actual_productivity'] >= df['targeted_productivity']).astype(int)
print("\nClass distribution for Meets_Target:")
print(df['Meets_Target'].value_counts(normalize=True).round(4))

=== Summary Statistics ===
                        count       mean     median     min         max  \
team                   1197.0     6.4269     6.0000  1.0000     12.0000   
targeted_productivity  1197.0     0.7296     0.7500  0.0700      0.8000   
smv                    1197.0    15.0622    15.2600  2.9000     54.5600   
wip                     691.0  1190.4660  1039.0000  7.0000  23122.0000   
over_time              1197.0  4567.4603  3960.0000  0.0000  25920.0000   
incentive              1197.0    38.2105     0.0000  0.0000   3600.0000   
idle_time              1197.0     0.7302     0.0000  0.0000    300.0000   
idle_men               1197.0     0.3693     0.0000  0.0000     45.0000   
no_of_style_change     1197.0     0.1504     0.0000  0.0000      2.0000   
no_of_workers          1197.0    34.6099    34.0000  2.0000     89.0000   
actual_productivity    1197.0     0.7351     0.7733  0.2337      1.1204   

                             std  
team                      3.4640  
ta

In [3]:
print(df.groupby('department')['wip'].apply(lambda s: s.isnull().sum()))
print(df['department'].value_counts())

department
finishing     249
finishing     257
sweing          0
Name: wip, dtype: int64
department
sweing        691
finishing     257
finishing     249
Name: count, dtype: int64


In [4]:
import pandas as pd
import numpy as np
import time
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score,
    accuracy_score, precision_score, recall_score, f1_score
)

# Load raw dataset
df_raw = pd.read_csv('garments_worker_productivity.csv')

# Create targets
y_reg = df_raw['actual_productivity'].values
y_clf = (df_raw['actual_productivity'] >= df_raw['targeted_productivity']).astype(int).values

# Drop target and date for baseline features
X_raw = df_raw.drop(columns=['actual_productivity', 'date'])

# Single fixed train-test split (80-20)
indices = np.arange(len(df_raw))
train_idx, test_idx = train_test_split(indices, test_size=0.2, random_state=42, stratify=y_clf)

X_train_raw, X_test_raw = X_raw.iloc[train_idx].copy(), X_raw.iloc[test_idx].copy()
y_reg_train, y_reg_test = y_reg[train_idx], y_reg[test_idx]
y_clf_train, y_clf_test = y_clf[train_idx], y_clf[test_idx]

# ==========================================
# PART A: Scikit-learn Implementation
# ==========================================
cat_cols = ['quarter', 'department', 'day']
num_cols = [c for c in X_raw.columns if c not in cat_cols]

num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

cat_pipeline = Pipeline([
    ('onehot', OneHotEncoder(drop='first', sparse_output=False))
])

preprocessor = ColumnTransformer([
    ('num', num_pipeline, num_cols),
    ('cat', cat_pipeline, cat_cols)
])

X_train_sk = preprocessor.fit_transform(X_train_raw)
X_test_sk = preprocessor.transform(X_test_raw)

# 1. Scikit-learn Linear Regression
sk_lin = LinearRegression()
t0 = time.perf_counter()
sk_lin.fit(X_train_sk, y_reg_train)
sk_lin_train_time = time.perf_counter() - t0

t0 = time.perf_counter()
y_reg_pred_sk = sk_lin.predict(X_test_sk)
sk_lin_pred_time = time.perf_counter() - t0

sk_reg_metrics = {
    'MAE': mean_absolute_error(y_reg_test, y_reg_pred_sk),
    'RMSE': np.sqrt(mean_squared_error(y_reg_test, y_reg_pred_sk)),
    'R2': r2_score(y_reg_test, y_reg_pred_sk),
    'Train Time (ms)': sk_lin_train_time * 1000,
    'Pred Time (ms)': sk_lin_pred_time * 1000
}

# 2. Scikit-learn Logistic Regression
sk_log = LogisticRegression(max_iter=1000, random_state=42)
t0 = time.perf_counter()
sk_log.fit(X_train_sk, y_clf_train)
sk_log_train_time = time.perf_counter() - t0

t0 = time.perf_counter()
y_clf_pred_sk = sk_log.predict(X_test_sk)
sk_log_pred_time = time.perf_counter() - t0

sk_clf_metrics = {
    'Accuracy': accuracy_score(y_clf_test, y_clf_pred_sk),
    'Precision': precision_score(y_clf_test, y_clf_pred_sk),
    'Recall': recall_score(y_clf_test, y_clf_pred_sk),
    'F1': f1_score(y_clf_test, y_clf_pred_sk),
    'Train Time (ms)': sk_log_train_time * 1000,
    'Pred Time (ms)': sk_log_pred_time * 1000
}

print("Scikit-Learn Regression:", sk_reg_metrics)
print("Scikit-Learn Classification:", sk_clf_metrics)

Scikit-Learn Regression: {'MAE': 0.10727843302917218, 'RMSE': np.float64(0.1437402979105884), 'R2': 0.2885214314434412, 'Train Time (ms)': 112.80299299983199, 'Pred Time (ms)': 1.6664759998548107}
Scikit-Learn Classification: {'Accuracy': 0.7125, 'Precision': 0.7572815533980582, 'Recall': 0.8914285714285715, 'F1': 0.8188976377952756, 'Train Time (ms)': 212.73080700007085, 'Pred Time (ms)': 0.5965100001503743}


In [5]:
# ==========================================
# PART B: Manual From-Scratch Implementation (NumPy & Pandas Only)
# ==========================================

# 1. Manual Preprocessing (matching Part A exactly)
X_tr_df = X_raw.iloc[train_idx].copy()
X_te_df = X_raw.iloc[test_idx].copy()

# Manual Median Imputation on numerical columns (fit on train only)
medians = X_tr_df[num_cols].median()
X_tr_num = X_tr_df[num_cols].fillna(medians).values.astype(float)
X_te_num = X_te_df[num_cols].fillna(medians).values.astype(float)

# Manual Standard Scaling (fit on train only, population std ddof=0 to match StandardScaler)
means = np.mean(X_tr_num, axis=0)
stds = np.std(X_tr_num, axis=0, ddof=0)
stds[stds == 0] = 1.0
X_tr_num_scaled = (X_tr_num - means) / stds
X_te_num_scaled = (X_te_num - means) / stds

# Manual One-Hot Encoding on categorical columns
X_tr_cat = pd.get_dummies(X_tr_df[cat_cols], drop_first=True, dtype=float)
X_te_cat = pd.get_dummies(X_te_df[cat_cols], drop_first=True, dtype=float)
# Align columns
X_te_cat = X_te_cat.reindex(columns=X_tr_cat.columns, fill_value=0.0)

X_train_manual = np.hstack([X_tr_num_scaled, X_tr_cat.values])
X_test_manual = np.hstack([X_te_num_scaled, X_te_cat.values])

# Check that manual preprocessing matches sklearn preprocessing
print("Max diff between Sklearn and Manual preprocessed matrices:", np.max(np.abs(X_train_sk - X_train_manual)))

# Manual Evaluation Metrics
def manual_mae(y_true, y_pred):
    return np.mean(np.abs(y_true - y_pred))

def manual_rmse(y_true, y_pred):
    return np.sqrt(np.mean((y_true - y_pred) ** 2))

def manual_r2(y_true, y_pred):
    ss_res = np.sum((y_true - y_pred) ** 2)
    ss_tot = np.sum((y_true - np.mean(y_true)) ** 2)
    return 1.0 - (ss_res / ss_tot)

def manual_clf_metrics(y_true, y_pred):
    tp = np.sum((y_true == 1) & (y_pred == 1))
    tn = np.sum((y_true == 0) & (y_pred == 0))
    fp = np.sum((y_true == 0) & (y_pred == 1))
    fn = np.sum((y_true == 1) & (y_pred == 0))

    acc = (tp + tn) / len(y_true)
    prec = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    rec = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = (2 * prec * rec) / (prec + rec) if (prec + rec) > 0 else 0.0
    return acc, prec, rec, f1

# 2. Manual Linear Regression (Baseline: Closed-Form Normal Equation with pseudoinverse)
class ManualLinearRegression:
    def __init__(self):
        self.theta = None

    def fit(self, X, y):
        X_b = np.c_[np.ones((X.shape[0], 1)), X]
        self.theta = np.linalg.pinv(X_b.T @ X_b) @ (X_b.T @ y)
        return self

    def predict(self, X):
        X_b = np.c_[np.ones((X.shape[0], 1)), X]
        return X_b @ self.theta

man_lin = ManualLinearRegression()
t0 = time.perf_counter()
man_lin.fit(X_train_manual, y_reg_train)
man_lin_train_time = time.perf_counter() - t0

t0 = time.perf_counter()
y_reg_pred_man = man_lin.predict(X_test_manual)
man_lin_pred_time = time.perf_counter() - t0

man_reg_metrics = {
    'MAE': manual_mae(y_reg_test, y_reg_pred_man),
    'RMSE': manual_rmse(y_reg_test, y_reg_pred_man),
    'R2': manual_r2(y_reg_test, y_reg_pred_man),
    'Train Time (ms)': man_lin_train_time * 1000,
    'Pred Time (ms)': man_lin_pred_time * 1000
}

# 3. Manual Logistic Regression (Baseline: Batch Gradient Descent)
class ManualLogisticRegression:
    def __init__(self, lr=0.1, n_iters=1000, threshold=0.5):
        self.lr = lr
        self.n_iters = n_iters
        self.threshold = threshold
        self.weights = None
        self.bias = 0.0

    @staticmethod
    def _sigmoid(z):
        z = np.clip(z, -500, 500)
        return 1.0 / (1.0 + np.exp(-z))

    def fit(self, X, y):
        n_samples, n_features = X.shape
        self.weights = np.zeros(n_features)
        self.bias = 0.0

        for _ in range(self.n_iters):
            linear_pred = X @ self.weights + self.bias
            probs = self._sigmoid(linear_pred)
            error = probs - y
            dw = (1.0 / n_samples) * (X.T @ error)
            db = (1.0 / n_samples) * np.sum(error)
            self.weights -= self.lr * dw
            self.bias -= self.lr * db
        return self

    def predict_proba(self, X):
        return self._sigmoid(X @ self.weights + self.bias)

    def predict(self, X):
        return (self.predict_proba(X) >= self.threshold).astype(int)

man_log = ManualLogisticRegression(lr=0.1, n_iters=1000)
t0 = time.perf_counter()
man_log.fit(X_train_manual, y_clf_train)
man_log_train_time = time.perf_counter() - t0

t0 = time.perf_counter()
y_clf_pred_man = man_log.predict(X_test_manual)
man_log_pred_time = time.perf_counter() - t0

acc_m, prec_m, rec_m, f1_m = manual_clf_metrics(y_clf_test, y_clf_pred_man)
man_clf_metrics = {
    'Accuracy': acc_m,
    'Precision': prec_m,
    'Recall': rec_m,
    'F1': f1_m,
    'Train Time (ms)': man_log_train_time * 1000,
    'Pred Time (ms)': man_log_pred_time * 1000
}

print("Manual Regression:", man_reg_metrics)
print("Manual Classification:", man_clf_metrics)

Max diff between Sklearn and Manual preprocessed matrices: 0.0
Manual Regression: {'MAE': np.float64(0.1072784330291723), 'RMSE': np.float64(0.14374029791058826), 'R2': np.float64(0.28852143144344244), 'Train Time (ms)': 37.99632799996289, 'Pred Time (ms)': 0.3268059999754769}
Manual Classification: {'Accuracy': np.float64(0.7333333333333333), 'Precision': np.float64(0.7605633802816901), 'Recall': np.float64(0.9257142857142857), 'F1': np.float64(0.8350515463917527), 'Train Time (ms)': 86.54122200005077, 'Pred Time (ms)': 0.17632200001571618}


In [6]:
# ==========================================
# PART C: Optimized Manual Implementation (NumPy & Pandas Only)
# ==========================================

def build_optimized_features(df_input, fit_stats=None):
    df_c = df_input.copy()

    # 1. Clean categorical string typo ('finishing ' -> 'finishing')
    df_c['department'] = df_c['department'].str.strip().replace({'sweing': 'sewing'})

    # 2. Domain-aware WIP imputation (0 for finishing department) + log transform
    df_c['wip_missing'] = df_c['wip'].isna().astype(float)
    df_c['wip_filled'] = df_c['wip'].fillna(0.0)

    # 3. Log-transform skewed numeric features
    df_c['log_incentive'] = np.log1p(np.maximum(0, df_c['incentive']))
    df_c['log_wip'] = np.log1p(np.maximum(0, df_c['wip_filled']))
    df_c['log_over_time'] = np.log1p(np.maximum(0, df_c['over_time']))
    df_c['log_idle_time'] = np.log1p(np.maximum(0, df_c['idle_time']))
    df_c['has_incentive'] = (df_c['incentive'] > 0).astype(float)
    df_c['has_idle'] = (df_c['idle_men'] > 0).astype(float)

    # 4. Domain ratios and interactions
    df_c['smv_per_worker'] = df_c['smv'] / (df_c['no_of_workers'] + 1e-5)
    df_c['overtime_per_worker'] = df_c['over_time'] / (df_c['no_of_workers'] + 1e-5)
    df_c['target_x_incentive'] = df_c['targeted_productivity'] * df_c['log_incentive']
    df_c['target_sq'] = df_c['targeted_productivity'] ** 2
    df_c['is_sewing'] = (df_c['department'] == 'sewing').astype(float)
    df_c['sewing_x_incentive'] = df_c['is_sewing'] * df_c['log_incentive']
    df_c['sewing_x_target'] = df_c['is_sewing'] * df_c['targeted_productivity']
    df_c['finishing_x_smv'] = (1.0 - df_c['is_sewing']) * df_c['smv']

    # Categorical one-hot encoding (including 'team' as categorical)
    df_c['team_str'] = df_c['team'].astype(str)
    cat_cols_opt = ['quarter', 'department', 'day', 'team_str']
    num_cols_opt = [
        'targeted_productivity', 'smv', 'wip_filled', 'wip_missing',
        'over_time', 'incentive', 'idle_time', 'idle_men',
        'no_of_style_change', 'no_of_workers',
        'log_incentive', 'log_wip', 'log_over_time', 'log_idle_time',
        'has_incentive', 'has_idle', 'smv_per_worker', 'overtime_per_worker',
        'target_x_incentive', 'target_sq', 'sewing_x_incentive',
        'sewing_x_target', 'finishing_x_smv'
    ]

    X_num = df_c[num_cols_opt].values.astype(float)
    X_cat_df = pd.get_dummies(df_c[cat_cols_opt], drop_first=True, dtype=float)

    if fit_stats is None:
        means = np.mean(X_num, axis=0)
        stds = np.std(X_num, axis=0, ddof=0)
        stds[stds == 0] = 1.0
        cat_columns = X_cat_df.columns
        fit_stats = {'means': means, 'stds': stds, 'cat_columns': cat_columns}
    else:
        means = fit_stats['means']
        stds = fit_stats['stds']
        X_cat_df = X_cat_df.reindex(columns=fit_stats['cat_columns'], fill_value=0.0)

    X_num_scaled = (X_num - means) / stds
    X_out = np.hstack([X_num_scaled, X_cat_df.values])
    return X_out, fit_stats

X_train_opt, opt_stats = build_optimized_features(X_train_raw)
X_test_opt, _ = build_optimized_features(X_test_raw, opt_stats)

# 1. Optimized Manual Ridge Regression (Closed-form Cholesky/Solve with L2 penalty)
class OptimizedRidgeRegression:
    def __init__(self, alpha=10.0):
        self.alpha = alpha
        self.theta = None

    def fit(self, X, y):
        n_samples, n_features = X.shape
        X_b = np.c_[np.ones((n_samples, 1)), X]
        reg_matrix = self.alpha * np.eye(n_features + 1)
        reg_matrix[0, 0] = 0.0  # Do not regularize bias
        self.theta = np.linalg.solve(X_b.T @ X_b + reg_matrix, X_b.T @ y)
        return self

    def predict(self, X):
        X_b = np.c_[np.ones((X.shape[0], 1)), X]
        return X_b @ self.theta

opt_lin = OptimizedRidgeRegression(alpha=10.0)
t0 = time.perf_counter()
opt_lin.fit(X_train_opt, y_reg_train)
opt_lin_train_time = time.perf_counter() - t0

t0 = time.perf_counter()
y_reg_pred_opt = opt_lin.predict(X_test_opt)
opt_lin_pred_time = time.perf_counter() - t0

opt_reg_metrics = {
    'MAE': manual_mae(y_reg_test, y_reg_pred_opt),
    'RMSE': manual_rmse(y_reg_test, y_reg_pred_opt),
    'R2': manual_r2(y_reg_test, y_reg_pred_opt),
    'Train Time (ms)': opt_lin_train_time * 1000,
    'Pred Time (ms)': opt_lin_pred_time * 1000
}

# 2. Optimized Manual Logistic Regression (Newton-Raphson / IRLS with L2 Regularization & Early Stopping)
class OptimizedLogisticRegressionIRLS:
    def __init__(self, reg_lambda=1.0, max_iters=25, tol=1e-6, threshold=0.5):
        self.reg_lambda = reg_lambda
        self.max_iters = max_iters
        self.tol = tol
        self.threshold = threshold
        self.theta = None
        self.n_iter_ = 0

    @staticmethod
    def _sigmoid(z):
        z = np.clip(z, -500, 500)
        return 1.0 / (1.0 + np.exp(-z))

    def fit(self, X, y):
        n_samples, n_features = X.shape
        X_b = np.c_[np.ones((n_samples, 1)), X]
        self.theta = np.zeros(n_features + 1)
        reg_eye = self.reg_lambda * np.eye(n_features + 1)
        reg_eye[0, 0] = 0.0  # Do not regularize bias

        for i in range(self.max_iters):
            self.n_iter_ = i + 1
            p = self._sigmoid(X_b @ self.theta)
            w = np.maximum(p * (1.0 - p), 1e-6)
            grad = X_b.T @ (p - y) + reg_eye @ self.theta
            # Vectorized Hessian: X_b.T @ diag(w) @ X_b
            H = (X_b.T * w) @ X_b + reg_eye
            step = np.linalg.solve(H, grad)
            self.theta -= step
            if np.linalg.norm(step, ord=np.inf) < self.tol:
                break
        return self

    def predict_proba(self, X):
        X_b = np.c_[np.ones((X.shape[0], 1)), X]
        return self._sigmoid(X_b @ self.theta)

    def predict(self, X):
        return (self.predict_proba(X) >= self.threshold).astype(int)

opt_log = OptimizedLogisticRegressionIRLS(reg_lambda=2.0, max_iters=25, tol=1e-5, threshold=0.5)
t0 = time.perf_counter()
opt_log.fit(X_train_opt, y_clf_train)
opt_log_train_time = time.perf_counter() - t0

t0 = time.perf_counter()
y_clf_pred_opt = opt_log.predict(X_test_opt)
opt_log_pred_time = time.perf_counter() - t0

acc_o, prec_o, rec_o, f1_o = manual_clf_metrics(y_clf_test, y_clf_pred_opt)
opt_clf_metrics = {
    'Accuracy': acc_o,
    'Precision': prec_o,
    'Recall': rec_o,
    'F1': f1_o,
    'Train Time (ms)': opt_log_train_time * 1000,
    'Pred Time (ms)': opt_log_pred_time * 1000,
    'Iterations': opt_log.n_iter_
}

print("Optimized Regression:", opt_reg_metrics)
print("Optimized Classification:", opt_clf_metrics)

Optimized Regression: {'MAE': np.float64(0.0987923126649563), 'RMSE': np.float64(0.13798958727073496), 'R2': np.float64(0.344311792917221), 'Train Time (ms)': 27.879425000037372, 'Pred Time (ms)': 0.2610849999200582}
Optimized Classification: {'Accuracy': np.float64(0.7125), 'Precision': np.float64(0.7880434782608695), 'Recall': np.float64(0.8285714285714286), 'F1': np.float64(0.807799442896936), 'Train Time (ms)': 204.87058899993826, 'Pred Time (ms)': 0.3400709999823448, 'Iterations': 8}


In [7]:
# Tune threshold and regularization on training set via 5-fold manual CV
def manual_cv_tune(X_tr, y_tr):
    n = len(y_tr)
    k = 5
    fold_size = n // k
    best_score = -1
    best_params = None
    for reg in [0.5, 1.0, 2.0, 5.0, 10.0, 20.0, 50.0]:
        for thresh in [0.40, 0.45, 0.48, 0.50, 0.52, 0.55]:
            accs, f1s = [], []
            for f in range(k):
                val_mask = np.zeros(n, dtype=bool)
                val_mask[f*fold_size:(f+1)*fold_size] = True
                model = OptimizedLogisticRegressionIRLS(reg_lambda=reg, max_iters=25, tol=1e-5, threshold=thresh)
                model.fit(X_tr[~val_mask], y_tr[~val_mask])
                preds = model.predict(X_tr[val_mask])
                acc, prec, rec, f1 = manual_clf_metrics(y_tr[val_mask], preds)
                accs.append(acc)
                f1s.append(f1)
            mean_score = 0.5 * np.mean(accs) + 0.5 * np.mean(f1s)
            if mean_score > best_score:
                best_score = mean_score
                best_params = (reg, thresh, np.mean(accs), np.mean(f1s))
    return best_params

best_reg, best_thresh, cv_acc, cv_f1 = manual_cv_tune(X_train_opt, y_clf_train)
print(f"Best CV params: reg_lambda={best_reg}, threshold={best_thresh} (CV Acc={cv_acc:.4f}, CV F1={cv_f1:.4f})")

opt_log = OptimizedLogisticRegressionIRLS(reg_lambda=best_reg, max_iters=25, tol=1e-5, threshold=best_thresh)
t0 = time.perf_counter()
opt_log.fit(X_train_opt, y_clf_train)
opt_log_train_time = time.perf_counter() - t0

t0 = time.perf_counter()
y_clf_pred_opt = opt_log.predict(X_test_opt)
opt_log_pred_time = time.perf_counter() - t0

acc_o, prec_o, rec_o, f1_o = manual_clf_metrics(y_clf_test, y_clf_pred_opt)
print(f"Test Metrics with tuned params: Acc={acc_o:.4f}, Prec={prec_o:.4f}, Rec={rec_o:.4f}, F1={f1_o:.4f}, Train Time={opt_log_train_time*1000:.3f}ms")

Best CV params: reg_lambda=1.0, threshold=0.45 (CV Acc=0.8073, CV F1=0.8749)
Test Metrics with tuned params: Acc=0.7000, Prec=0.7725, Rec=0.8343, F1=0.8022, Train Time=8.366ms


In [8]:
def build_clean_features(df_input, fit_stats=None):
    df_c = df_input.copy()

    # 1. Clean categorical string typo ('finishing ' -> 'finishing')
    df_c['department'] = df_c['department'].str.strip().replace({'sweing': 'sewing'})

    # 2. Domain-aware WIP imputation (0 for finishing department) + log transform
    df_c['wip_filled'] = df_c['wip'].fillna(0.0)

    # 3. Focused domain feature engineering (non-collinear)
    df_c['log_incentive'] = np.log1p(np.maximum(0, df_c['incentive']))
    df_c['has_incentive'] = (df_c['incentive'] > 0).astype(float)
    df_c['has_idle'] = (df_c['idle_men'] > 0).astype(float)
    df_c['smv_per_worker'] = df_c['smv'] / (df_c['no_of_workers'] + 1e-5)
    df_c['target_sq'] = df_c['targeted_productivity'] ** 2
    df_c['is_sewing'] = (df_c['department'] == 'sewing').astype(float)
    df_c['sewing_x_incentive'] = df_c['is_sewing'] * df_c['log_incentive']

    cat_cols_opt = ['quarter', 'department', 'day']
    num_cols_opt = [
        'team', 'targeted_productivity', 'smv', 'wip_filled',
        'over_time', 'incentive', 'idle_time', 'idle_men',
        'no_of_style_change', 'no_of_workers',
        'log_incentive', 'has_incentive', 'has_idle',
        'smv_per_worker', 'target_sq', 'sewing_x_incentive'
    ]

    X_num = df_c[num_cols_opt].values.astype(float)
    X_cat_df = pd.get_dummies(df_c[cat_cols_opt], drop_first=True, dtype=float)

    if fit_stats is None:
        means = np.mean(X_num, axis=0)
        stds = np.std(X_num, axis=0, ddof=0)
        stds[stds == 0] = 1.0
        cat_columns = X_cat_df.columns
        fit_stats = {'means': means, 'stds': stds, 'cat_columns': cat_columns}
    else:
        means = fit_stats['means']
        stds = fit_stats['stds']
        X_cat_df = X_cat_df.reindex(columns=fit_stats['cat_columns'], fill_value=0.0)

    X_num_scaled = (X_num - means) / stds
    X_out = np.hstack([X_num_scaled, X_cat_df.values])
    return X_out, fit_stats

X_tr_clean, clean_stats = build_clean_features(X_train_raw)
X_te_clean, _ = build_clean_features(X_test_raw, clean_stats)

# Test Ridge Regression
for alpha in [1.0, 5.0, 10.0, 20.0]:
    r_mod = OptimizedRidgeRegression(alpha=alpha).fit(X_tr_clean, y_reg_train)
    p_reg = r_mod.predict(X_te_clean)
    print(f"Ridge alpha={alpha}: MAE={manual_mae(y_reg_test, p_reg):.4f}, RMSE={manual_rmse(y_reg_test, p_reg):.4f}, R2={manual_r2(y_reg_test, p_reg):.4f}")

# Test IRLS Logistic Regression
for reg in [1.0, 5.0, 10.0, 25.0, 50.0]:
    for th in [0.45, 0.48, 0.50]:
        c_mod = OptimizedLogisticRegressionIRLS(reg_lambda=reg, max_iters=25, tol=1e-5, threshold=th).fit(X_tr_clean, y_clf_train)
        p_clf = c_mod.predict(X_te_clean)
        acc, prec, rec, f1 = manual_clf_metrics(y_clf_test, p_clf)
        print(f"LogReg reg={reg}, th={th}: Acc={acc:.4f}, Prec={prec:.4f}, Rec={rec:.4f}, F1={f1:.4f}, Iters={c_mod.n_iter_}")

Ridge alpha=1.0: MAE=0.0972, RMSE=0.1357, R2=0.3663
Ridge alpha=5.0: MAE=0.0980, RMSE=0.1359, R2=0.3638
Ridge alpha=10.0: MAE=0.0988, RMSE=0.1365, R2=0.3588
Ridge alpha=20.0: MAE=0.0998, RMSE=0.1374, R2=0.3496
LogReg reg=1.0, th=0.45: Acc=0.7000, Prec=0.7512, Rec=0.8800, F1=0.8105, Iters=7
LogReg reg=1.0, th=0.48: Acc=0.6958, Prec=0.7602, Rec=0.8514, F1=0.8032, Iters=7
LogReg reg=1.0, th=0.5: Acc=0.7000, Prec=0.7641, Rec=0.8514, F1=0.8054, Iters=7
LogReg reg=5.0, th=0.45: Acc=0.6917, Prec=0.7440, Rec=0.8800, F1=0.8063, Iters=7
LogReg reg=5.0, th=0.48: Acc=0.6833, Prec=0.7463, Rec=0.8571, F1=0.7979, Iters=7
LogReg reg=5.0, th=0.5: Acc=0.7000, Prec=0.7696, Rec=0.8400, F1=0.8033, Iters=7
LogReg reg=10.0, th=0.45: Acc=0.6833, Prec=0.7391, Rec=0.8743, F1=0.8010, Iters=7
LogReg reg=10.0, th=0.48: Acc=0.6875, Prec=0.7475, Rec=0.8629, F1=0.8011, Iters=7
LogReg reg=10.0, th=0.5: Acc=0.6958, Prec=0.7629, Rec=0.8457, F1=0.8022, Iters=7
LogReg reg=25.0, th=0.45: Acc=0.6917, Prec=0.7349, Rec=0.9029

In [9]:
# Check IRLS on X_train_manual across reg_lambda
for reg in [0.1, 1.0, 5.0, 10.0, 20.0, 50.0, 100.0]:
    for th in [0.45, 0.48, 0.50]:
        c_mod = OptimizedLogisticRegressionIRLS(reg_lambda=reg, max_iters=25, tol=1e-5, threshold=th).fit(X_train_manual, y_clf_train)
        p_clf = c_mod.predict(X_test_manual)
        acc, prec, rec, f1 = manual_clf_metrics(y_clf_test, p_clf)
        if acc >= 0.73:
            print(f"X_train_manual IRLS reg={reg}, th={th}: Acc={acc:.4f}, Prec={prec:.4f}, Rec={rec:.4f}, F1={f1:.4f}, Iters={c_mod.n_iter_}")

# Now test adding features one by one to see which ones boost classification AND regression!

X_train_manual IRLS reg=5.0, th=0.5: Acc=0.7333, Prec=0.7581, Rec=0.9314, F1=0.8359, Iters=7
X_train_manual IRLS reg=10.0, th=0.48: Acc=0.7375, Prec=0.7478, Rec=0.9657, F1=0.8429, Iters=6
X_train_manual IRLS reg=20.0, th=0.45: Acc=0.7333, Prec=0.7342, Rec=0.9943, F1=0.8447, Iters=6
X_train_manual IRLS reg=50.0, th=0.45: Acc=0.7375, Prec=0.7353, Rec=1.0000, F1=0.8475, Iters=5
X_train_manual IRLS reg=50.0, th=0.48: Acc=0.7375, Prec=0.7353, Rec=1.0000, F1=0.8475, Iters=5
X_train_manual IRLS reg=50.0, th=0.5: Acc=0.7375, Prec=0.7353, Rec=1.0000, F1=0.8475, Iters=5
X_train_manual IRLS reg=100.0, th=0.45: Acc=0.7375, Prec=0.7353, Rec=1.0000, F1=0.8475, Iters=5
X_train_manual IRLS reg=100.0, th=0.48: Acc=0.7375, Prec=0.7353, Rec=1.0000, F1=0.8475, Iters=5
X_train_manual IRLS reg=100.0, th=0.5: Acc=0.7375, Prec=0.7353, Rec=1.0000, F1=0.8475, Iters=5


In [10]:
# Let's test which feature additions improve BOTH Regression and Classification
base_tr = X_raw.iloc[train_idx].copy()
base_te = X_raw.iloc[test_idx].copy()

def test_feature_set(extra_fn, name):
    tr = extra_fn(base_tr.copy())
    te = extra_fn(base_te.copy())
    c_cols = [c for c in tr.select_dtypes(include='object').columns]
    n_cols = [c for c in tr.columns if c not in c_cols]
    med = tr[n_cols].median()
    X_tr_n = tr[n_cols].fillna(med).values.astype(float)
    X_te_n = te[n_cols].fillna(med).values.astype(float)
    m = X_tr_n.mean(axis=0)
    s = X_tr_n.std(axis=0, ddof=0)
    s[s==0] = 1.0
    X_tr_n = (X_tr_n - m) / s
    X_te_n = (X_te_n - m) / s
    X_tr_c = pd.get_dummies(tr[c_cols], drop_first=True, dtype=float)
    X_te_c = pd.get_dummies(te[c_cols], drop_first=True, dtype=float).reindex(columns=X_tr_c.columns, fill_value=0.0)
    X_tr = np.hstack([X_tr_n, X_tr_c.values])
    X_te = np.hstack([X_te_n, X_te_c.values])

    r_mod = OptimizedRidgeRegression(alpha=5.0).fit(X_tr, y_reg_train)
    p_reg = r_mod.predict(X_te)

    c_mod = OptimizedLogisticRegressionIRLS(reg_lambda=5.0, max_iters=25, tol=1e-5, threshold=0.5).fit(X_tr, y_clf_train)
    p_clf = c_mod.predict(X_te)
    acc, prec, rec, f1 = manual_clf_metrics(y_clf_test, p_clf)
    print(f"{name:30s} | R2={manual_r2(y_reg_test, p_reg):.4f}, RMSE={manual_rmse(y_reg_test, p_reg):.4f} | Acc={acc:.4f}, Prec={prec:.4f}, Rec={rec:.4f}, F1={f1:.4f}")

test_feature_set(lambda d: d, "Baseline")
test_feature_set(lambda d: d.assign(department=d['department'].str.strip().replace({'sweing': 'sewing'})), "Clean dept")
test_feature_set(lambda d: d.assign(
    department=d['department'].str.strip().replace({'sweing': 'sewing'}),
    has_incentive=(d['incentive'] > 0).astype(float),
    smv_per_worker=d['smv'] / (d['no_of_workers'] + 1e-5),
    target_sq=d['targeted_productivity'] ** 2
), "Clean + domain features")
test_feature_set(lambda d: d.assign(
    department=d['department'].str.strip().replace({'sweing': 'sewing'}),
    has_incentive=(d['incentive'] > 0).astype(float),
    smv_per_worker=d['smv'] / (d['no_of_workers'] + 1e-5),
    target_sq=d['targeted_productivity'] ** 2,
    team_str=d['team'].astype(str)
), "Clean + domain + team OHE")

Baseline                       | R2=0.2891, RMSE=0.1437 | Acc=0.7333, Prec=0.7581, Rec=0.9314, F1=0.8359
Clean dept                     | R2=0.2693, RMSE=0.1457 | Acc=0.7167, Prec=0.7465, Rec=0.9257, F1=0.8265
Clean + domain features        | R2=0.3159, RMSE=0.1409 | Acc=0.6875, Prec=0.7525, Rec=0.8514, F1=0.7989
Clean + domain + team OHE      | R2=0.3229, RMSE=0.1402 | Acc=0.6792, Prec=0.7722, Rec=0.7943, F1=0.7831


In [11]:
print(df_raw.groupby('department')[['actual_productivity', 'targeted_productivity', 'incentive', 'smv', 'no_of_workers']].mean())
print(df_raw.groupby('department')['Meets_Target' if 'Meets_Target' in df_raw else 'actual_productivity'].count())
df_raw['Meets_Target'] = (df_raw['actual_productivity'] >= df_raw['targeted_productivity']).astype(int)
print(df_raw.groupby('department')['Meets_Target'].mean())
print(df_raw.groupby(['department', 'quarter']).size())

            actual_productivity  targeted_productivity  incentive        smv  \
department                                                                     
finishing              0.722876               0.731325  60.240964   3.979719   
finishing              0.782089               0.743191   0.000000   3.797237   
sweing                 0.722013               0.723980  44.483357  23.245412   

            no_of_workers  
department                 
finishing       10.397590  
finishing       10.112840  
sweing          52.445731  
department
finishing     249
finishing     257
sweing        691
Name: actual_productivity, dtype: int64
department
finishing     0.542169
finishing     0.649805
sweing        0.829233
Name: Meets_Target, dtype: float64
department  quarter 
finishing   Quarter1     73
            Quarter2     85
            Quarter3     40
            Quarter4     51
finishing   Quarter1     76
            Quarter2     62
            Quarter3     41
            Quarter4  